# Task1 寒武纪行情 demo walkthrough

这份 notebook 用教学方式复盘 Task1：数据从哪里来、前复权怎么计算、网页怎么生成。Notebook 不复制长脚本，只展示关键步骤；完整实现见 `scripts/`。

## Goal

Task1 要做的是一个最小数据产品：

1. 从 TuShare 获取寒武纪日线行情和复权因子。
2. 计算前复权价格，并保存三份 CSV。
3. 用 matplotlib 生成价格图。
4. 写出一个静态网页 `web/index.html`。

## Setup

项目环境由根目录 `pyproject.toml` 和 `uv.lock` 管理：

```powershell
uv sync --group dev
```

如果只是学习流程，可以直接读取已经保存好的 CSV，不需要 TuShare token。

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
TASK_DIR = ROOT / "Task1" if (ROOT / "Task1").exists() else ROOT
SCRIPTS = TASK_DIR / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from build_site import build_site

DATA_DIR = TASK_DIR / "data"
WEB_DIR = TASK_DIR / "web"

## Steps

### 1. 读取已经保存的 combined CSV

`combined` 文件把未复权价格、复权因子、前复权价格合在一张表里，最适合后续展示。

In [2]:
combined_csv = sorted(DATA_DIR.glob("cambricon_688256_SH_daily_combined_*.csv"))[-1]
df = pd.read_csv(combined_csv, parse_dates=["trade_date"])
df.head()

,ts_code,trade_date,date,open,high,low,close,pre_close,change,pct_chg,vol,amount,adj_factor,qfq_open,qfq_high,qfq_low,qfq_close,qfq_pre_close,qfq_change,qfq_pct_chg
0,688256.SH,2025-07-04,2025-07-04,545.00,557.97,538.67,547.47,547.30,0.17,0.0311,56859.33,3111587.176,1.0,365.477468,374.175161,361.232564,367.133852,NaN,NaN,NaN
1,688256.SH,2025-07-07,2025-07-07,544.00,553.86,541.01,541.38,547.47,-6.09,-1.1124,33310.55,1810291.915,1.0,364.806867,371.418991,362.801770,363.049893,367.133852,-4.083959,-1.112390
2,688256.SH,2025-07-08,2025-07-08,540.00,545.30,538.10,542.77,541.38,1.39,0.2568,39527.81,2143484.029,1.0,362.124464,365.678648,360.850322,363.982028,363.049893,0.932135,0.256751
3,688256.SH,2025-07-09,2025-07-09,544.00,544.78,533.50,535.00,542.77,-7.77,-1.4315,49069.59,2643083.782,1.0,364.806867,365.329936,357.765558,358.771459,363.982028,-5.210569,-1.431546
4,688256.SH,2025-07-10,2025-07-10,534.01,537.87,520.67,523.50,535.00,-11.50,-2.1495,85815.16,4522151.657,1.0,358.107564,360.696084,349.161749,351.059549,358.771459,-7.711910,-2.149533


### 2. 做一个最小摘要

这里不需要复杂统计，只要确认日期区间、最新价格和区间收益即可。

In [3]:
first, latest = df.iloc[0], df.iloc[-1]
summary = {
    "rows": len(df),
    "start": first["date"],
    "end": latest["date"],
    "latest_close": latest["close"],
    "qfq_return": latest["qfq_close"] / first["qfq_close"] - 1,
}
summary

{'rows': 242,
 'start': '2025-07-04',
 'end': '2026-07-03',
 'latest_close': np.float64(1353.0),
 'qfq_return': np.float64(2.685304400241109)}

### 3. 理解前复权公式

复权的目的是把分红、送转等因素调整进历史价格。Task1 使用的公式是：

```text
前复权价格 = 未复权价格 * 当日复权因子 / 区间最新复权因子
```

In [4]:
demo = df[["date", "close", "adj_factor", "qfq_close"]].head(3).copy()
latest_factor = df["adj_factor"].iloc[-1]
demo["manual_qfq_close"] = demo["close"] * demo["adj_factor"] / latest_factor
demo

,date,close,adj_factor,qfq_close,manual_qfq_close
0,2025-07-04,547.47,1.0,367.133852,367.133852
1,2025-07-07,541.38,1.0,363.049893,363.049893
2,2025-07-08,542.77,1.0,363.982028,363.982028


### 4. 画出价格走势

实际网页由 `scripts/build_site.py` 完成。这里用几行代码展示核心图形是什么。

In [5]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["trade_date"], df["close"], label="未复权收盘价")
ax.plot(df["trade_date"], df["qfq_close"], label="前复权收盘价")
ax.set_title("寒武纪收盘价走势")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

C:\Users\13377\AppData\Local\Temp\ipykernel_11604\3215625407.py:7: UserWarning: Glyph 23506 (\N{CJK UNIFIED IDEOGRAPH-5BD2}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_11604\3215625407.py:7: UserWarning: Glyph 27494 (\N{CJK UNIFIED IDEOGRAPH-6B66}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_11604\3215625407.py:7: UserWarning: Glyph 32426 (\N{CJK UNIFIED IDEOGRAPH-7EAA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_11604\3215625407.py:7: UserWarning: Glyph 25910 (\N{CJK UNIFIED IDEOGRAPH-6536}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_11604\3215625407.py:7: UserWarning: Glyph 30424 (\N{CJK UNIFIED IDEOGRAPH-76D8}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_11604\3215625407.py:7: UserWarning: Glyph 20215 (\N{CJK 

### 5. 生成网页

代码拆分后，网页生成只需要读取 CSV、画图、写 HTML。

In [6]:
html_path = build_site(combined_csv)
html_path

WindowsPath('C:/Users/13377/Desktop/PKU-WorkShop-202607/Task1/web/index.html')

## Checks

确认网页和图片都存在。

In [7]:
assert html_path.exists()
assert (WEB_DIR / "cambricon_price.png").exists()
"Task1 walkthrough checks passed"

'Task1 walkthrough checks passed'

## Next Steps

- 需要刷新数据时，运行 `uv run python .\Task1\scripts\run_all.py --start-date ... --end-date ...`。
- 只改网页样式时，运行 `uv run python .\Task1\scripts\run_all.py --skip-fetch`。